<a href="https://colab.research.google.com/github/Lingjelleima/ai_for_early_cervical_cancer_screening/blob/coding_section/cervical_pipeline_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-source anomaly detection for cervical screening under unpaired data (v2)

**Research idea.** Cytology images, colposcopy images and clinical-history tables come from different
sources and share no patient IDs, so a model that learns from all three jointly cannot be trained or validated.
This notebook implements three *independent* normative anomaly-detection pipelines
(ResNet-50 -> ViT-S distillation + AE/IF/OCSVM for the two image modalities, IF/OCSVM for history),
scored on a common robust scale (Eq. 7) so that a missingness-aware late-fusion stage can be plugged in once a
linked cohort exists. **No patient-level fusion is computed here.**

**Run on a T4 GPU** runtime (not a TPU), top to bottom.

### What changed relative to v1 (the notebook that produced the numbers reported in the paper)
Numerical behaviour of cells 0-12 is unchanged (same splits, seeds, models, hyper-parameters), so v1 and v2 should agree up to GPU non-determinism.
Changes are limited to: (i) environment/version logging; (ii) guards (missing archives, degenerate MAD, stale distillation cache);
(iii) extra reporting columns (test-set prevalence, AP lift over prevalence, per-stratum operating points, split counts);
(iv) honest naming: bootstrap clusters are *patients* only for DAI colposcopy and the clinical table, otherwise *images/parent images*;
(v) contamination study now logs every scorer, and the cross-dataset test also reports a crop/field-stratified matrix;
(vi) two optional, clearly marked **[NEW]** analyses at the end (label-assisted scorer selection on the labeled-validation split;
repeated group-level re-splitting). Their results are *not* in the paper's v1 tables.


In [ ]:
# 0. Setup + GPU check + reproducibility record
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "kagglehub"], check=True)
import random, numpy as np, pandas as pd, torch, timm, sklearn
assert torch.cuda.is_available(), ("No GPU. Runtime > Change runtime type > T4 GPU "
                                   "(not v5e/v6e TPU), Save, then re-run from the top.")
print("GPU:", torch.cuda.get_device_name(0))
random.seed(0); np.random.seed(0); torch.manual_seed(0)   # every stochastic component below is additionally seeded explicitly
ENV = dict(python=sys.version.split()[0], torch=torch.__version__, timm=timm.__version__, sklearn=sklearn.__version__,
           numpy=np.__version__, pandas=pd.__version__, gpu=torch.cuda.get_device_name(0))
print(ENV)                                                 # copy into the paper's reproducibility statement
# Bootstrap resamples. Defaults equal the values used for the reported (v1) results; raise to 1000 for camera-ready CIs.
N_BOOT_AUROC, N_BOOT_PAIRED, N_BOOT_CLIN = 200, 300, 300


GPU: Tesla T4
{'python': '3.13.15', 'torch': '2.11.0+cu130', 'timm': '1.0.30', 'sklearn': '1.6.1', 'numpy': '2.1.3', 'pandas': '2.2.3', 'gpu': 'Tesla T4'}


In [ ]:
# 1. Mount Drive, unzip datasets, locate roots by folder name
import os, glob, hashlib, json, time, warnings
import numpy as np, pandas as pd
from PIL import Image
warnings.filterwarnings("ignore")
from google.colab import drive
drive.mount("/content/drive")
D = "/content/drive/MyDrive"
SAVE = f"{D}/cervical_run"; os.makedirs(SAVE, exist_ok=True)

def unzip(zp, out):
    if os.path.isdir(out) and os.listdir(out): return
    os.makedirs(out, exist_ok=True)
    subprocess.run(["unzip", "-q", "-o", zp, "-d", out], check=True)

def unzip_nested(root):
    for _ in range(3):
        zs = [os.path.join(dp, f) for dp, _, fs in os.walk(root) for f in fs if f.lower().endswith(".zip")]
        todo = [z for z in zs if not os.path.isdir(z[:-4] + "__x")]
        if not todo: break
        for z in todo: unzip(z, z[:-4] + "__x")

ZIPS = {  # local name -> zip file in your Drive
    "risk":    "cervical+cancer+risk+factors.zip",
    "malhari": "Malhari Dataset.zip",
    "apcem":   "APCEM-Enhanced Cervical Cytology Image Dataset.zip",
    "dai":     "Datasets for Artificial Intelligence (AI)-assisted.zip",
}
missing = [z for z in ZIPS.values() if not os.path.exists(f"{D}/{z}")]
assert not missing, f"Upload these archives to {D} first: {missing}"
for k, z in ZIPS.items():
    unzip(f"{D}/{z}", f"/content/{k}"); unzip_nested(f"/content/{k}")

import kagglehub
SIP_RAW = kagglehub.dataset_download("prahladmehandiratta/cervical-cancer-largest-dataset-sipakmed")

def find_dir(root, name):
    for dp, dirs, _ in os.walk(root):
        if name in dirs: return os.path.join(dp, name)
    raise FileNotFoundError(f"no folder named {name!r} under {root}")

APC    = os.path.dirname(find_dir("/content/apcem", "NILM"))
MALPAP = find_dir("/content/malhari", "pap")
DAI    = os.path.dirname(find_dir("/content/dai", "01_Normal"))
SIP    = os.path.dirname(find_dir(SIP_RAW, "im_Parabasal"))
for n, p in dict(APC=APC, MALPAP=MALPAP, DAI=DAI, SIP=SIP).items():
    print(f"{n:7s}", p, "->", sorted(os.listdir(p))[:12])

Mounted at /content/drive


100%|██████████| 6.40G/6.40G [01:01<00:00, 112MB/s]

Extracting files...


APC     /content/apcem/APCEM-Enhanced Cervical Cytology Image Dataset/APCEM-Enhanced Cervical Cytology Image Dataset__x/Enhanced Cervical Cytology Image dataset -> ['ASCH', 'ASCUS', 'HSIL', 'LSIL', 'NILM', 'SCC']
MALPAP  /content/malhari/Malhari Dataset/Malhari_Dataset_V1/pap -> ['HSIL', 'LSIL', 'NILM', 'SSC']
DAI     /content/dai/Datasets for Artificial Intelligence (AI)-assisted/New folder/Colposcopy_Data_In_Brief-20251130T051652Z-1-001-iTMAEm__x/Colposcopy_Data_In_Brief -> ['.DS_Store', '01_Normal', '02_Abnormal']
SIP     /root/.cache/kagglehub/datasets/prahladmehandiratta/cervical-cancer-largest-dataset-sipakmed/versions/1 -> ['im_Dyskeratotic', 'im_Koilocytotic', 'im_Metaplastic', 'im_Parabasal', 'im_Superficial-Intermediate']


In [ ]:
# 2. Manifest: reference (normal) cohorts vs abnormal cohorts
IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")

def collect(folder, source, modality, label, severity, gfn):
    assert os.path.isdir(folder), f"Missing folder: {folder}"
    rows = [dict(path=os.path.join(dp, f), source=source, modality=modality, label=label, severity=severity)
            for dp, _, fs in os.walk(folder) for f in fs if f.lower().endswith(IMG_EXT)]
    assert rows, f"No images in {folder}"
    d = pd.DataFrame(rows); d["group"] = [gfn(p) for p in d.path]; return d

# Leakage groups: patient folder (DAI); parent-image id (SIPaKMeD: a field image and its crops stay together);
# otherwise each image is its own group (APCEM/Malhari have no patient or slide IDs). Only DAI is patient-level.
g_file = lambda src: (lambda p: f"{src}:{p}")
g_dai  = lambda p: f"DAI:{os.path.relpath(p, DAI).split(os.sep)[1]}"
def g_sip(p):
    cls = os.path.relpath(p, SIP).split(os.sep)[0]
    return f"SipakMed:{cls}:{os.path.splitext(os.path.basename(p))[0].split('_')[0]}"

normals = pd.concat([
    collect(f"{APC}/NILM",                         "APCEM",    "cytology",   0, "normal", g_file("APCEM")),
    collect(f"{MALPAP}/NILM",                      "Malhari",  "cytology",   0, "normal", g_file("Malhari")),
    collect(f"{SIP}/im_Superficial-Intermediate",  "SipakMed", "cytology",   0, "normal", g_sip),
    collect(f"{SIP}/im_Parabasal",                 "SipakMed", "cytology",   0, "normal", g_sip),
    collect(f"{DAI}/01_Normal",                    "DAI",      "colposcopy", 0, "normal", g_dai),
], ignore_index=True)

abn = [collect(f"{APC}/{c}", "APCEM", "cytology", 1, c, g_file("APCEM")) for c in ["ASCUS", "ASCH", "LSIL", "HSIL", "SCC"]]
abn += [collect(f"{MALPAP}/{c}", "Malhari", "cytology", 1, c, g_file("Malhari")) for c in ["LSIL", "HSIL", "SSC"]]
abn += [collect(f"{SIP}/im_{c}", "SipakMed", "cytology", 1, c, g_sip) for c in ["Dyskeratotic", "Koilocytotic"]]  # Metaplastic excluded on purpose
abn += [collect(f"{DAI}/02_Abnormal", "DAI", "colposcopy", 1, "abnormal", g_dai)]
# Malhari colposcopy (CIN1-3) is NOT used: it has no normal class, so a DAI-normal vs Malhari-CIN test would measure dataset differences.
abnormals = pd.concat(abn, ignore_index=True)

nrm_p = {os.path.relpath(p, DAI).split(os.sep)[1] for p in normals[normals.source == "DAI"].path}
abn_p = {os.path.relpath(p, DAI).split(os.sep)[1] for p in abnormals[abnormals.source == "DAI"].path}
print("DAI patients present in both classes (must be 0):", len(nrm_p & abn_p))
for d in (normals, abnormals):
    d["group_kind"] = np.where(d.source == "DAI", "patient", np.where(d.source == "SipakMed", "parent-image", "image"))
_dai = normals[normals.source == "DAI"].path
print("DAI path structure (is magnification / pre-post acetic-acid encoded in the folders?):",
      [os.path.relpath(p, DAI).split(os.sep) for p in _dai.iloc[:3]])
print("DAI folder names below patient level:", sorted({t for p in _dai for t in os.path.relpath(p, DAI).split(os.sep)[2:-1]})[:20])


DAI patients present in both classes (must be 0): 0
DAI path structure (is magnification / pre-post acetic-acid encoded in the folders?): [['01_Normal', 'buth_130', 'p2.jpg'], ['01_Normal', 'buth_130', 'po2.jpg'], ['01_Normal', 'buth_130', 'p4.jpg']]
DAI folder names below patient level: []


In [ ]:
# 3. Four partitions per modality (paper Section V-A), split by patient/slide group
SEED = 42
def split_groups(df, fr, seed=SEED):
    g = df.group.unique().copy(); np.random.default_rng(seed).shuffle(g)
    parts = np.split(g, (np.cumsum(fr)[:-1] * len(g)).astype(int))
    return [df[df.group.isin(p)] for p in parts]

parts = []
for src, g in normals.groupby("source"):          # every source appears in every split
    for name, d in zip(["train", "val", "lval", "test"], split_groups(g, [0.55, 0.15, 0.10, 0.20])):
        parts.append(d.assign(split=name))        # train=reference-training, val=validation-reference, lval=labeled validation (normal part)
for src, g in abnormals.groupby("source"):        # abnormal images never enter reference-training or validation-reference
    for name, d in zip(["lval", "pool", "test"], split_groups(g, [0.20, 0.20, 0.60])):
        parts.append(d.assign(split=name))        # pool = used ONLY by the contamination study
meta = pd.concat(parts, ignore_index=True)
meta["img_type"] = np.where((meta.source == "SipakMed") & meta.path.str.upper().str.contains("CROPPED"), "crop", "field")

for m in ["cytology", "colposcopy"]:                  # sanity: no group in two splits
    d = meta[meta.modality == m]
    assert d.groupby("group").split.nunique().max() == 1, "group leaked across splits"
cnt = meta.groupby(["modality", "source", "split", "label"]).size().unstack(fill_value=0)
print(cnt)
for (m, s, sp), r in cnt.groupby(level=[0, 1, 2]):
    if sp in ("train", "val", "lval", "test"): assert r.values.sum() > 0, f"empty split {m}/{s}/{sp}"
cnt.to_csv(f"{SAVE}/split_counts.csv")
tst = meta[meta.split == "test"].groupby(["modality", "source", "img_type"]).label.agg(n="size", n_abn="sum")
tst["abn_share"] = (tst.n_abn / tst.n).round(3)
print("\nTest-set composition. NOTE: average precision of a random scorer equals abn_share, so read AP only as lift over it:")
print(tst)


label                         0     1
modality   source   split            
colposcopy DAI      lval    211   226
                    pool      0   244
                    test    435   752
                    train  1167     0
                    val     321     0
cytology   APCEM    lval     78   694
                    pool      0   695
                    test    156  2085
                    train   428     0
                    val     117     0
           Malhari  lval     16    32
                    pool      0    32
                    test     32    96
                    train    86     0
                    val      24     0
           SipakMed lval    165   442
                    pool      0   433
                    test    358  1224
                    train  1087     0
                    val     242     0

Test-set composition. NOTE: average precision of a random scorer equals abn_share, so read AP only as lift over it:
                                 n  n_abn  abn_

In [ ]:
# 4. Shrink every image to 224x224 once (local disk), in parallel
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
SM = "/content/small"; os.makedirs(SM, exist_ok=True)

def shrink(p):
    out = f"{SM}/{hashlib.md5(p.encode()).hexdigest()}.jpg"
    if not os.path.exists(out):
        try:
            im = Image.open(p); im.draft("RGB", (448, 448))
            im.convert("RGB").resize((224, 224), Image.BILINEAR, reducing_gap=2.0).save(out, quality=95)
        except Exception as e:
            print("skip", p, e); return ""
    return out

t0 = time.time()
with ThreadPoolExecutor(16) as ex:
    meta["sp"] = list(tqdm(ex.map(shrink, meta.path), total=len(meta)))
meta = meta[meta.sp != ""].reset_index(drop=True)
meta.to_csv(f"{SAVE}/meta.csv", index=False)
print(f"shrunk {len(meta)} images in {time.time()-t0:.0f}s")

  0%|          | 0/11878 [00:00<?, ?it/s]

shrunk 11878 images in 75s


In [ ]:
# 5. Frozen pretrained features (one pass, cached). resnet50 = the paper's TEACHER features
import timm, torch.nn.functional as Fnn
from torch.utils.data import Dataset, DataLoader
dev = "cuda"
BB = {"resnet50":    "resnet50.tv_in1k",
      "vit_s_in21k": "vit_small_patch16_224.augreg_in21k_ft_in1k",
      "dino_vits":   "vit_small_patch16_224.dino"}
CACHE = f"{SAVE}/features.npz"

class SmallDS(Dataset):
    def __init__(s, paths): s.p = list(paths)
    def __len__(s): return len(s.p)
    def __getitem__(s, i):
        return torch.from_numpy(np.asarray(Image.open(s.p[i]).convert("RGB")).copy()).permute(2, 0, 1)

FE = None
if os.path.exists(CACHE):
    z = np.load(CACHE, allow_pickle=True)
    if len(z["sp"]) == len(meta) and (z["sp"] == meta.sp.values.astype(str)).all():
        FE = {k: z[k] for k in BB}; print("loaded cached features")
if FE is None:
    models, mean, std = {}, {}, {}
    for k, name in BB.items():
        m = timm.create_model(name, pretrained=True, num_classes=0).eval().to(dev)
        cfg = timm.data.resolve_data_config({}, model=m)
        models[k] = m
        print(f"  {k}: {sum(p.numel() for p in m.parameters())/1e6:.1f} M parameters")
        mean[k] = torch.tensor(cfg["mean"], device=dev).view(1, 3, 1, 1)
        std[k]  = torch.tensor(cfg["std"],  device=dev).view(1, 3, 1, 1)
    outs = {k: [] for k in BB}
    dl = DataLoader(SmallDS(meta.sp), batch_size=128, num_workers=2, pin_memory=True)
    t0 = time.time()
    with torch.no_grad():
        for x in tqdm(dl):
            x = x.to(dev, non_blocking=True).float().div_(255)
            for k, m in models.items():
                with torch.autocast("cuda", dtype=torch.float16):
                    f = m((x - mean[k]) / std[k])
                outs[k].append(Fnn.normalize(f.float(), dim=1).cpu())
    FE = {k: torch.cat(v).numpy() for k, v in outs.items()}
    np.savez(CACHE, sp=meta.sp.values.astype(str), **FE)
    print(f"extracted in {time.time()-t0:.0f}s", {k: v.shape for k, v in FE.items()})
    del models; torch.cuda.empty_cache()

model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

  resnet50: 23.5 M parameters


model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 86.7MB            

model.safetensors: downloading bytes:           |  0.00B            

  dino_vits: 21.7 M parameters


  0%|          | 0/93 [00:00<?, ?it/s]

extracted in 38s {'resnet50': (11878, 2048), 'vit_s_in21k': (11878, 384), 'dino_vits': (11878, 384)}


In [ ]:
# 5b. Visual representation learning exactly as in the paper: frozen ResNet-50 teacher -> ViT student (Eq. 1)
# Student = ViT-S + projection head to R^2048, trained on reference-training NORMAL images of ONE modality only.
# Teacher targets are the cached, L2-normalised ResNet-50 features (the teacher is frozen, so this is identical to
# running it online; the only consequence is that no image augmentation is used during distillation).
DIST_EPOCHS, DIST_MAXN, DIST_LR = 5, 8000, 1e-4      # small budget on purpose: a few minutes per modality
DCACHE = f"{SAVE}/distilled.npz"
STUDENT = "vit_small_patch16_224.augreg_in21k_ft_in1k"   # NOTE: the student is itself ImageNet-21k-pretrained (disclosed in the paper)
DCFG = f"{STUDENT}|{DIST_EPOCHS}|{DIST_MAXN}|{DIST_LR}"       # cache key: any change to these invalidates the cached student embeddings
tz = FE["resnet50"]
SP = meta.sp.values

class DistDS(Dataset):
    def __init__(s, idx): s.idx = np.asarray(idx)
    def __len__(s): return len(s.idx)
    def __getitem__(s, i):
        j = s.idx[i]
        im = torch.from_numpy(np.asarray(Image.open(SP[j]).convert("RGB")).copy()).permute(2, 0, 1)
        return im, torch.from_numpy(tz[j])

def train_student(m, seed=0):
    torch.manual_seed(seed)
    tr = meta[(meta.modality == m) & (meta.split == "train")]
    idx = tr.sample(min(DIST_MAXN, len(tr)), random_state=seed).index.values
    st = timm.create_model(STUDENT, pretrained=True, num_classes=2048).to(dev)
    print(f"  student: {sum(p.numel() for p in st.parameters())/1e6:.1f} M parameters")
    cfg = timm.data.resolve_data_config({}, model=st)
    mean = torch.tensor(cfg["mean"], device=dev).view(1, 3, 1, 1); std = torch.tensor(cfg["std"], device=dev).view(1, 3, 1, 1)
    opt = torch.optim.AdamW(st.parameters(), lr=DIST_LR, weight_decay=0.05)
    scaler = torch.cuda.amp.GradScaler()
    dl = DataLoader(DistDS(idx), batch_size=64, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    for ep in range(DIST_EPOCHS):
        st.train(); tot = nb = 0
        for x, t in dl:
            x = x.to(dev, non_blocking=True).float().div_(255); t = t.to(dev)
            with torch.autocast("cuda", dtype=torch.float16):
                s = st((x - mean) / std)
            s = Fnn.normalize(s.float(), dim=1)
            loss = ((s - t) ** 2).sum(1).mean()                      # Eq. 1
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item(); nb += 1
        print(f"  [{m}] epoch {ep+1}/{DIST_EPOCHS}  L_distill = {tot/nb:.4f}")
    st.eval()
    rows = meta.index[meta.modality == m].values                      # teacher is discarded; student embeds every image of this modality
    outs = []
    with torch.no_grad():
        for x, _ in DataLoader(DistDS(rows), batch_size=128, num_workers=2, pin_memory=True):
            x = x.to(dev, non_blocking=True).float().div_(255)
            with torch.autocast("cuda", dtype=torch.float16):
                s = st((x - mean) / std)
            outs.append(Fnn.normalize(s.float(), dim=1).cpu())
    return torch.cat(outs).numpy(), rows

Dz = None
if os.path.exists(DCACHE):
    z = np.load(DCACHE, allow_pickle=True)
    if len(z["sp"]) == len(meta) and (z["sp"] == meta.sp.values.astype(str)).all() and "cfg" in z.files and str(z["cfg"]) == DCFG:
        Dz = z["distilled"]; print("loaded cached distilled features")
if Dz is None:
    Dz = np.zeros((len(meta), 2048), np.float32); t0 = time.time()
    for m in ["cytology", "colposcopy"]:
        out, rows = train_student(m); Dz[rows] = out
        v = meta.index[(meta.modality == m) & (meta.split == "val")].values      # held-out normals: how well does the student mimic the teacher?
        print(f"  [{m}] mean cosine(student, teacher) on unseen validation normals: {(Dz[v]*tz[v]).sum(1).mean():.3f}")
    np.savez(DCACHE, sp=meta.sp.values.astype(str), cfg=DCFG, distilled=Dz)
    print(f"distillation done in {time.time()-t0:.0f}s"); torch.cuda.empty_cache()
FE = {"distilled": Dz, **FE}                      # 'distilled' is the paper's encoder; the others are the ablation / baselines

  student: 22.5 M parameters
  [cytology] epoch 1/5  L_distill = 0.8936
  [cytology] epoch 2/5  L_distill = 0.3481
  [cytology] epoch 3/5  L_distill = 0.2815
  [cytology] epoch 4/5  L_distill = 0.2463
  [cytology] epoch 5/5  L_distill = 0.2279
  [cytology] mean cosine(student, teacher) on unseen validation normals: 0.886
  student: 22.5 M parameters
  [colposcopy] epoch 1/5  L_distill = 0.9630
  [colposcopy] epoch 2/5  L_distill = 0.2864
  [colposcopy] epoch 3/5  L_distill = 0.2217
  [colposcopy] epoch 4/5  L_distill = 0.2053
  [colposcopy] epoch 5/5  L_distill = 0.1902
  [colposcopy] mean cosine(student, teacher) on unseen validation normals: 0.908
distillation done in 60s


In [ ]:
# 6a. Autoencoder f_theta: R^d -> R^128, g_phi: R^128 -> R^d, trained on reference embeddings only (Eq. 2)
import torch.nn as nn
class AEModel:
    def __init__(self, seed=0, hidden=512, latent=128, epochs=60):
        self.seed, self.h, self.zdim, self.E = seed, hidden, latent, epochs
    def _t(self, Z): return torch.tensor(np.asarray(Z, np.float32) * self.k, device=dev)
    def fit(self, Z):
        torch.manual_seed(self.seed)
        d = Z.shape[1]; self.k = float(np.sqrt(d))       # constant rescale of unit-norm z (removed again by Eq. 7 normalisation)
        self.enc = nn.Sequential(nn.Linear(d, self.h), nn.ReLU(), nn.Linear(self.h, self.zdim)).to(dev)
        self.dec = nn.Sequential(nn.Linear(self.zdim, self.h), nn.ReLU(), nn.Linear(self.h, d)).to(dev)
        opt = torch.optim.Adam(list(self.enc.parameters()) + list(self.dec.parameters()), lr=1e-3, weight_decay=1e-5)
        X = self._t(Z); n = len(X)
        for _ in range(self.E):
            perm = torch.randperm(n, device=dev)
            for i in range(0, n, 256):
                xb = X[perm[i:i + 256]]
                loss = Fnn.mse_loss(self.dec(self.enc(xb)), xb)
                opt.zero_grad(); loss.backward(); opt.step()
        self.enc.eval(); self.dec.eval(); return self
    @torch.no_grad()
    def encode(self, Z): return self.enc(self._t(Z)).cpu().numpy()
    @torch.no_grad()
    def rec_err(self, Z):                                 # S_rec = ||z - g(f(z))||^2 (Eq. 3)
        X = self._t(Z); return (((self.dec(self.enc(X)) - X) ** 2).sum(1) / self.k ** 2).cpu().numpy()

In [ ]:
# 6b. Normative detectors, Eq. 7 normalisation, Eq. 8-9 aggregation (+ Mahalanobis / kNN baselines)
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import NearestNeighbors
from sklearn.covariance import LedoitWolf

def robust(s_val):                      # Eq. 7: median/MAD of the validation-reference scores, not clipped
    med = np.median(s_val); mad = np.median(np.abs(s_val - med))
    if mad < 1e-6: print("WARNING: MAD of validation scores ~ 0, Eq. 7 normalisation is degenerate")
    return lambda s: (s - med) / (1.4826 * mad + 1e-8)

class Bundle:
    def __init__(self, seed=0, ae_epochs=60):
        self.seed, self.E = seed, ae_epochs
    def fit(self, Ztr, Zva):
        rng = np.random.default_rng(self.seed)
        self.ae = AEModel(seed=self.seed, epochs=self.E).fit(Ztr)
        Ltr = self.ae.encode(Ztr); self.mu = Ltr.mean(0)                    # reference latent centroid
        self.iforest = IsolationForest(n_estimators=200, random_state=self.seed).fit(Ltr)       # Eq. 5, on latents
        sub = rng.choice(len(Ltr), min(5000, len(Ltr)), replace=False)                           # OCSVM is O(n^2)
        self.ocsvm = OneClassSVM(kernel="rbf", gamma="scale", nu=0.05).fit(Ltr[sub])             # Eq. 6, on latents
        sub = rng.choice(len(Ztr), min(10000, len(Ztr)), replace=False)
        self.lw = LedoitWolf().fit(Ztr[sub]); self.nn = NearestNeighbors(n_neighbors=10).fit(Ztr[sub])
        rv = self._raw(Zva)
        self.n = {k: robust(v) for k, v in rv.items()}                      # statistics from validation-reference only
        nv = {k: self.n[k](v) for k, v in rv.items()}
        self.n["ae"] = robust((nv["rec"] + nv["lat"]) / 2)
        return self
    def _raw(self, Z):
        L = self.ae.encode(Z)
        return {"rec": self.ae.rec_err(Z), "lat": ((L - self.mu) ** 2).sum(1),
                "if": -self.iforest.score_samples(L), "oc": -self.ocsvm.decision_function(L),     # larger = more abnormal
                "maha": self.lw.mahalanobis(Z), "knn": self.nn.kneighbors(Z)[0].mean(1)}
    def score(self, Z):
        r = self._raw(Z); s = {k: self.n[k](v) for k, v in r.items()}
        s["ae"] = self.n["ae"]((s["rec"] + s["lat"]) / 2)                   # Eq. 8, equal weights
        s["img"] = (s["ae"] + s["if"] + s["oc"]) / 3                        # Eq. 9, equal weights
        s["ae_if"] = (s["ae"] + s["if"]) / 2; s["ae_oc"] = (s["ae"] + s["oc"]) / 2   # ablation combos (Section V-D)
        return s

In [ ]:
# 7. Metrics (V-B), thresholds (V-C), group-level cluster bootstrap (patients for DAI/clinical, images or parent images otherwise), paired comparison
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

def cluster_idx(groups):
    _, inv = np.unique(groups, return_inverse=True)
    order = np.argsort(inv); return np.split(order, np.cumsum(np.bincount(inv))[:-1])

def auroc_ci(y, s, groups, n_boot=N_BOOT_AUROC, seed=0):
    rng = np.random.default_rng(seed); spl = cluster_idx(groups); vals = []
    for _ in range(n_boot):
        ii = np.concatenate([spl[j] for j in rng.integers(0, len(spl), len(spl))])
        if len(set(y[ii])) == 2: vals.append(roc_auc_score(y[ii], s[ii]))
    return tuple(np.percentile(vals, [2.5, 97.5])) if vals else (np.nan, np.nan)

def evaluate(te, sc, ci_for=("img",)):
    rows = []
    for (src, it), g in te.groupby(["source", "img_type"]):
        y = g.label.values
        if len(set(y)) < 2: continue
        ix = te.index.get_indexer(g.index)
        for k, s in sc.items():
            s = s[ix]; lo = hi = np.nan
            if k in ci_for: lo, hi = auroc_ci(y, s, g.group.values)
            rows.append(dict(source=src, img_type=it, n=len(g), n_abn=int(y.sum()), prevalence=float(y.mean()), scorer=k,
                             auroc=roc_auc_score(y, s), ap=average_precision_score(y, s),
                             ap_lift=average_precision_score(y, s) / float(y.mean()), ci_lo=lo, ci_hi=hi))
    return pd.DataFrame(rows)

def macro(df): return df.groupby("scorer").auroc.mean()           # mean over strata so no stratum dominates

def mod_data(m):
    d = meta[meta.modality == m]
    return {k: d[d.split == k] for k in ["train", "val", "lval", "test", "pool"]}

def thresholds(s_valref, y_lv, s_lv, alpha=0.05):                 # Section V-C
    fpr, tpr, th = roc_curve(y_lv, s_lv)
    return {"valref_q95": float(np.quantile(s_valref, 1 - alpha)),   # (1-alpha) quantile of validation-reference scores
            "youden": float(th[np.argmax(tpr - fpr)])}               # maximises J = TPR - FPR on labeled validation

def op_metrics(y, s, thr):
    p = s >= thr
    tp = int((p & (y == 1)).sum()); fp = int((p & (y == 0)).sum()); fn = int((~p & (y == 1)).sum()); tn = int((~p & (y == 0)).sum())
    sens = tp / max(tp + fn, 1); spec = tn / max(tn + fp, 1); prec = tp / max(tp + fp, 1)
    return dict(sens=sens, spec=spec, fpr=1 - spec, fnr=1 - sens, f1=2 * prec * sens / max(prec + sens, 1e-12))

def sens_at_spec(y, s, spec=0.95):
    fpr, tpr, _ = roc_curve(y, s); return float(tpr[fpr <= 1 - spec].max())

def operating_rows(te, s, thr, **tag):
    rows = []
    for (src, it), g in te.groupby(["source", "img_type"]):
        y = g.label.values
        if len(set(y)) < 2: continue
        ss = s[te.index.get_indexer(g.index)]; base = dict(source=src, img_type=it, sens_at_95spec=sens_at_spec(y, ss), **tag)
        for name, t in thr.items(): rows.append({**base, "threshold": name, **op_metrics(y, ss, t)})
    return rows

def paired_diff(te, sa, sb, n_boot=N_BOOT_PAIRED, seed=0):                  # paired cluster bootstrap on AUROC(a) - AUROC(b)
    rng = np.random.default_rng(seed); rows = []
    for (src, it), g in te.groupby(["source", "img_type"]):
        y = g.label.values
        if len(set(y)) < 2: continue
        ix = te.index.get_indexer(g.index); a, b = sa[ix], sb[ix]; spl = cluster_idx(g.group.values)
        d0 = roc_auc_score(y, a) - roc_auc_score(y, b); ds = []
        for _ in range(n_boot):
            ii = np.concatenate([spl[j] for j in rng.integers(0, len(spl), len(spl))])
            if len(set(y[ii])) == 2: ds.append(roc_auc_score(y[ii], a[ii]) - roc_auc_score(y[ii], b[ii]))
        lo, hi = np.percentile(ds, [2.5, 97.5])
        rows.append(dict(source=src, img_type=it, diff=d0, ci_lo=lo, ci_hi=hi, ci_excludes_0=bool(lo > 0 or hi < 0)))
    return pd.DataFrame(rows)

In [ ]:
# 8. Source-shortcut check (can the embedding tell which dataset an image came from?)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
tr_c = mod_data("cytology")["train"]
for b in FE:
    idx = tr_c.sample(min(4000, len(tr_c)), random_state=0).index.values
    acc = cross_val_score(LogisticRegression(max_iter=2000), FE[b][idx], meta.source.values[idx], cv=3).mean()
    maj = meta.source.values[idx].tolist(); maj = max(map(maj.count, set(maj))) / len(maj)
    print(f"{b:12s} source predictable: {acc:.3f} (majority baseline {maj:.3f})")
print("If predictability is ~1.0, read per-source results only, never a pooled number.")

distilled    source predictable: 0.941 (majority baseline 0.679)
resnet50     source predictable: 0.968 (majority baseline 0.679)
vit_s_in21k  source predictable: 0.983 (majority baseline 0.679)
dino_vits    source predictable: 0.989 (majority baseline 0.679)
If predictability is ~1.0, read per-source results only, never a pooled number.


In [ ]:
# 9. MAIN EXPERIMENT + component ablation + operating points (backbones x modalities x seeds)
SEEDS = (0, 1, 2)
main_rows, ops_rows, SC, LV, TS = [], [], {}, {}, {}   # LV/TS keep labeled-validation / test scores for the [NEW] selection analysis
t0 = time.time()
for b in FE:
    for m in ["cytology", "colposcopy"]:
        Dm = mod_data(m)
        Z = {k: FE[b][v.index.values] for k, v in Dm.items()}
        for sd in SEEDS:
            bun = Bundle(seed=sd).fit(Z["train"], Z["val"])
            sc = bun.score(Z["test"]); LV[(b, m, sd)] = bun.score(Z["lval"]); TS[(b, m, sd)] = sc
            thr = thresholds(bun.score(Z["val"])["img"], Dm["lval"].label.values, bun.score(Z["lval"])["img"])
            main_rows.append(evaluate(Dm["test"], sc, ci_for=("img",) if sd == 0 else ()).assign(backbone=b, modality=m, seed=sd))
            ops_rows += operating_rows(Dm["test"], sc["img"], thr, backbone=b, modality=m, seed=sd)
            if sd == 0: SC[(b, m)] = sc["img"]
        print(f"{b} / {m} done ({time.time()-t0:.0f}s)")
main = pd.concat(main_rows, ignore_index=True); main.to_csv(f"{SAVE}/main_results.csv", index=False)
ops = pd.DataFrame(ops_rows); ops.to_csv(f"{SAVE}/operating_points.csv", index=False)

summary = (main.groupby(["backbone", "modality", "source", "img_type", "scorer"]).auroc
           .agg(["mean", "std"]).round(3).reset_index())
print("\nPer-source AUROC of the full ensemble S_img (mean/std over seeds):")
print(summary[summary.scorer == "img"].to_string(index=False))
ci0 = main[(main.seed == 0) & (main.scorer == "img")][["backbone", "modality", "source", "img_type", "auroc", "ap", "prevalence", "ci_lo", "ci_hi"]].round(3)
print("\nSeed-0 AUROC / AP (AP must be compared with prevalence) with group-level bootstrap 95% CI:"); print(ci0.to_string(index=False))
abl = (main.groupby(["backbone", "modality", "scorer", "seed"]).auroc.mean().groupby(["backbone", "modality", "scorer"]).mean()
       .unstack("scorer").round(3))
print("\nComponent ablation (macro AUROC over strata, mean over seeds):"); print(abl.to_string())
print("\nOperating points of S_img (mean over strata and seeds):")
print(ops.groupby(["backbone", "modality", "threshold"])[["sens", "spec", "fpr", "fnr", "f1", "sens_at_95spec"]].mean().round(3).to_string())
print("\nOperating points per stratum (seed 0, S_img):")
print(ops[ops.seed == 0].groupby(["backbone", "modality", "source", "img_type", "threshold"])[["sens", "spec", "sens_at_95spec"]].mean().round(3).to_string())


distilled / cytology done (102s)
distilled / colposcopy done (148s)
resnet50 / cytology done (253s)
resnet50 / colposcopy done (299s)
vit_s_in21k / cytology done (309s)
vit_s_in21k / colposcopy done (316s)
dino_vits / cytology done (326s)
dino_vits / colposcopy done (332s)

Per-source AUROC of the full ensemble S_img (mean/std over seeds):
   backbone   modality   source img_type scorer  mean   std
  dino_vits colposcopy      DAI    field    img 0.467 0.005
  dino_vits   cytology    APCEM    field    img 0.746 0.028
  dino_vits   cytology  Malhari    field    img 0.612 0.010
  dino_vits   cytology SipakMed     crop    img 0.953 0.016
  dino_vits   cytology SipakMed    field    img 0.770 0.016
  distilled colposcopy      DAI    field    img 0.489 0.001
  distilled   cytology    APCEM    field    img 0.654 0.021
  distilled   cytology  Malhari    field    img 0.591 0.019
  distilled   cytology SipakMed     crop    img 0.705 0.030
  distilled   cytology SipakMed    field    img 0.654 0.06

In [ ]:
# 9b. Paired comparison: distilled student vs teacher features / other backbones (same test images)
for m in ["cytology", "colposcopy"]:
    te = mod_data(m)["test"]
    for other in [k for k in FE if k != "distilled"]:
        r = paired_diff(te, SC[("distilled", m)], SC[(other, m)])
        print(f"\n{m}: AUROC(distilled) - AUROC({other})"); print(r.round(3).to_string(index=False))


cytology: AUROC(distilled) - AUROC(resnet50)
  source img_type   diff  ci_lo  ci_hi  ci_excludes_0
   APCEM    field -0.040 -0.086  0.002          False
 Malhari    field -0.111 -0.227 -0.008           True
SipakMed     crop -0.193 -0.263 -0.125           True
SipakMed    field -0.123 -0.216 -0.025           True

cytology: AUROC(distilled) - AUROC(vit_s_in21k)
  source img_type   diff  ci_lo  ci_hi  ci_excludes_0
   APCEM    field  0.029 -0.034  0.083          False
 Malhari    field  0.258  0.141  0.377           True
SipakMed     crop -0.031 -0.120  0.049          False
SipakMed    field -0.092 -0.198  0.020          False

cytology: AUROC(distilled) - AUROC(dino_vits)
  source img_type   diff  ci_lo  ci_hi  ci_excludes_0
   APCEM    field -0.073 -0.125 -0.022           True
 Malhari    field -0.022 -0.106  0.071          False
SipakMed     crop -0.262 -0.358 -0.182           True
SipakMed    field -0.062 -0.194  0.070          False

colposcopy: AUROC(distilled) - AUROC(resnet50)


In [ ]:
# 10. Contamination study: add abnormal images to the reference-training set (Section V-E)
BMAIN = "distilled"        # NOTE: the student is distilled on clean normals; only the detectors see the contaminated reference set
crows = []
for m in ["cytology", "colposcopy"]:
    Dm = mod_data(m); tr, va, te, pool = Dm["train"], Dm["val"], Dm["test"], Dm["pool"]
    for frac in [0.0, 0.01, 0.02, 0.05, 0.10]:
        for sd in SEEDS:
            n_add = int(round(frac * len(tr) / (1 - frac))) if frac else 0
            add = pool.sample(min(n_add, len(pool)), random_state=sd) if n_add else pool.iloc[:0]
            Ztr = np.vstack([FE[BMAIN][tr.index.values], FE[BMAIN][add.index.values]])
            bun = Bundle(seed=sd).fit(Ztr, FE[BMAIN][va.index.values])
            r = evaluate(te, bun.score(FE[BMAIN][te.index.values]), ci_for=())
            mac = macro(r)
            crows.append(dict(modality=m, frac=frac, actual=len(add) / (len(tr) + len(add)), seed=sd,
                              auroc_img=mac["img"], auroc_maha=mac["maha"],
                              **{f"auroc_{k}": v for k, v in mac.items() if k not in ("img", "maha")}))
contam = pd.DataFrame(crows); contam.to_csv(f"{SAVE}/contamination.csv", index=False)
print(contam.groupby(["modality", "frac"])[["auroc_img", "auroc_maha", "auroc_rec", "auroc_knn"]].agg(["mean", "std"]).round(3).to_string())

                auroc_img        auroc_maha        auroc_rec        auroc_knn       
                     mean    std       mean    std      mean    std      mean    std
modality   frac                                                                     
colposcopy 0.00     0.489  0.001      0.482  0.000     0.494  0.007     0.483  0.000
           0.01     0.488  0.001      0.482  0.001     0.488  0.003     0.483  0.000
           0.02     0.488  0.001      0.482  0.000     0.488  0.002     0.483  0.000
           0.05     0.488  0.002      0.480  0.000     0.488  0.003     0.482  0.001
           0.10     0.489  0.001      0.479  0.000     0.491  0.003     0.481  0.001
cytology   0.00     0.651  0.021      0.787  0.000     0.880  0.012     0.908  0.000
           0.01     0.660  0.020      0.794  0.005     0.877  0.002     0.906  0.001
           0.02     0.627  0.024      0.802  0.009     0.874  0.007     0.903  0.001
           0.05     0.609  0.005      0.799  0.004     0.860  0.0

In [ ]:
# 11. Cross-dataset test (cytology): detectors trained on one source's normals, tested on every source
XB = "resnet50"            # teacher features: no learned representation, so no other source can leak in through the encoder
Dm = mod_data("cytology"); tr, va, te = Dm["train"], Dm["val"], Dm["test"]
mat, S_cross = {}, {}
for src in ["APCEM", "Malhari", "SipakMed"]:
    t_, v_ = tr[tr.source == src], va[va.source == src]
    bun = Bundle(seed=0).fit(FE[XB][t_.index.values], FE[XB][v_.index.values])
    s = bun.score(FE[XB][te.index.values])["img"]; S_cross[src] = s
    mat[src] = {tgt: (roc_auc_score(g.label, s[te.index.get_indexer(g.index)]) if g.label.nunique() == 2 else np.nan)
                for tgt, g in te.groupby("source")}
cross = pd.DataFrame(mat).T.round(3); cross.index.name = "train \\ test"
cross.to_csv(f"{SAVE}/cross_dataset.csv"); print(cross.to_string())
# Same matrix but stratified by image type (SipakMed crops and fields are pooled in the matrix above)
mat2 = {}
for src_tr, s in S_cross.items():
    mat2[src_tr] = {f"{tgt}/{it}": (roc_auc_score(g.label, s[te.index.get_indexer(g.index)]) if g.label.nunique() == 2 else np.nan)
                    for (tgt, it), g in te.groupby(["source", "img_type"])}
cross2 = pd.DataFrame(mat2).T.round(3); cross2.index.name = "train \\ test"
cross2.to_csv(f"{SAVE}/cross_dataset_stratified.csv"); print(); print(cross2.to_string())


              APCEM  Malhari  SipakMed
train \ test                          
APCEM         0.672    0.639     0.659
Malhari       0.623    0.484     0.751
SipakMed      0.570    0.695     0.893

              APCEM/field  Malhari/field  SipakMed/crop  SipakMed/field
train \ test                                                           
APCEM               0.672          0.639          0.782           0.720
Malhari             0.623          0.484          0.813           0.643
SipakMed            0.570          0.695          0.911           0.841


In [ ]:
# 12. Clinical-history branch (UCI risk-factor table), fully independent of the images
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
csvs = sorted(glob.glob("/content/risk/**/*.csv", recursive=True), key=os.path.getsize, reverse=True)
df = pd.read_csv(csvs[0]).replace("?", np.nan); df.columns = [c.strip() for c in df.columns]
FEATS = [c for c in ["Age", "Number of sexual partners", "First sexual intercourse", "Num of pregnancies",
                     "Smokes", "Smokes (years)", "Smokes (packs/year)", "Hormonal Contraceptives",
                     "Hormonal Contraceptives (years)", "IUD", "IUD (years)", "STDs", "STDs (number)"] if c in df.columns]
OUTS = [c for c in ["Biopsy", "Citology", "Hinselmann", "Schiller"] if c in df.columns]
X = df[FEATS].apply(pd.to_numeric, errors="coerce"); out = df[OUTS].apply(pd.to_numeric, errors="coerce")
ref = (out.fillna(1) == 0).all(axis=1).values        # reference = all screening outcomes negative (missing counts as not-reference)
pos = (out["Biopsy"] == 1).values
print("features:", FEATS, "| reference:", ref.sum(), "| biopsy-positive:", pos.sum())
print(f"records: {len(df)} | excluded from BOTH classes (a screening test positive but Biopsy negative, or outcome missing): {len(df) - ref.sum() - pos.sum()}")

crow = []
ridx = np.where(ref)[0]
for sd in SEEDS:
    r = np.random.default_rng(sd).permutation(ridx)
    a, b_ = int(.6 * len(r)), int(.75 * len(r))
    tr_i, va_i, te_n = r[:a], r[a:b_], r[b_:]
    te_i = np.concatenate([te_n, np.where(pos)[0]])
    y = np.r_[np.zeros(len(te_n)), np.ones(pos.sum())]
    if sd == 0: print(f"clinical test set: {len(te_n)} reference normals + {int(pos.sum())} biopsy-positive; prevalence {y.mean():.3f} (AP of a random scorer)")
    pre = Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]).fit(X.iloc[tr_i])   # reference-training statistics only
    A, V, T = (pre.transform(X.iloc[i]) for i in (tr_i, va_i, te_i))
    iff = IsolationForest(n_estimators=300, random_state=sd).fit(A)
    oc = OneClassSVM(kernel="rbf", gamma="scale", nu=0.05).fit(A)
    raw = lambda Z: {"if": -iff.score_samples(Z), "oc": -oc.decision_function(Z)}
    nrm = {k: robust(v) for k, v in raw(V).items()}
    s = {k: nrm[k](v) for k, v in raw(T).items()}; s["hist"] = (s["if"] + s["oc"]) / 2
    hv = (nrm["if"](raw(V)["if"]) + nrm["oc"](raw(V)["oc"])) / 2
    op = op_metrics(y, s["hist"], float(np.quantile(hv, 0.95)))        # validation-reference 95th-percentile threshold
    for k, v in s.items():
        lo, hi = auroc_ci(y, v, np.arange(len(y)), n_boot=N_BOOT_CLIN) if sd == 0 else (np.nan, np.nan)
        crow.append(dict(seed=sd, scorer=k, auroc=roc_auc_score(y, v), ap=average_precision_score(y, v), ci_lo=lo, ci_hi=hi,
                         **({f"op_{a}": b for a, b in op.items()} if k == "hist" else {})))
clin = pd.DataFrame(crow); clin.to_csv(f"{SAVE}/clinical.csv", index=False)
print(clin.groupby("scorer")[["auroc", "ap"]].agg(["mean", "std"]).round(3).to_string())
print(clin[(clin.seed == 0) & (clin.scorer == "hist")][["auroc", "ci_lo", "ci_hi"]].round(3).to_string(index=False))
print("AUROC near 0.5 => deviation in clinical history is not aligned with biopsy risk (the limitation in Section VII).")
print("\nAll tables saved to", SAVE)

features: ['Age', 'Number of sexual partners', 'First sexual intercourse', 'Num of pregnancies', 'Smokes', 'Smokes (years)', 'Smokes (packs/year)', 'Hormonal Contraceptives', 'Hormonal Contraceptives (years)', 'IUD', 'IUD (years)', 'STDs', 'STDs (number)'] | reference: 756 | biopsy-positive: 55
records: 858 | excluded from BOTH classes (a screening test positive but Biopsy negative, or outcome missing): 47
clinical test set: 189 reference normals + 55 biopsy-positive; prevalence 0.225 (AP of a random scorer)
        auroc            ap       
         mean    std   mean    std
scorer                            
hist    0.644  0.031  0.359  0.038
if      0.649  0.026  0.365  0.042
oc      0.600  0.023  0.329  0.027
 auroc  ci_lo  ci_hi
 0.611  0.508    0.7
AUROC near 0.5 => deviation in clinical history is not aligned with biopsy risk (the limitation in Section VII).

All tables saved to /content/drive/MyDrive/cervical_run


## Optional extensions **[NEW in v2 - not part of the reported v1 results]**
The next two cells answer the two most likely reviewer objections to the v1 study:
(A) *the equal-weight ensemble of Eq. 9 is dragged down by weak components - what if the scorer is chosen on the labeled-validation split, as the protocol allows?*
(B) *all results use one fixed split (seed 42) - how stable are they under repeated group-level re-splitting?*
Report (A) as **weakly supervised model selection** (abnormal labels used for selection only), never as an unsupervised result.

In [ ]:
# 13. [NEW] Label-assisted scorer selection on the LABELED-VALIDATION split, then one-shot evaluation on test.
# Abnormal labels are used only to CHOOSE among already-fitted unsupervised scorers (never to fit a detector or on test data).
CAND = ["rec", "lat", "ae", "if", "oc", "maha", "knn", "img"]
def macro_auc(df_, sc, k):
    v = []
    for _, g in df_.groupby(["source", "img_type"]):
        y = g.label.values
        if len(set(y)) == 2: v.append(roc_auc_score(y, sc[k][df_.index.get_indexer(g.index)]))
    return float(np.mean(v))
sel_rows = []
for (b, m, sd), scl in LV.items():
    Dm = mod_data(m)
    va = {k: macro_auc(Dm["lval"], scl, k) for k in CAND}
    best = max(va, key=va.get)
    sel_rows.append(dict(backbone=b, modality=m, seed=sd, selected=best, lval_auroc=va[best],
                         test_auroc=macro_auc(Dm["test"], TS[(b, m, sd)], best),
                         test_auroc_full_img=macro_auc(Dm["test"], TS[(b, m, sd)], "img")))
sel = pd.DataFrame(sel_rows); sel.to_csv(f"{SAVE}/scorer_selection.csv", index=False)
print("Macro AUROC: selected scorer vs the equal-weight ensemble S_img (mean over seeds)")
print(sel.groupby(["backbone", "modality"])[["lval_auroc", "test_auroc", "test_auroc_full_img"]].mean().round(3).to_string())
print(sel.groupby(["backbone", "modality"]).selected.agg(lambda s_: s_.value_counts().to_dict()).to_string())


Macro AUROC: selected scorer vs the equal-weight ensemble S_img (mean over seeds)
                        lval_auroc  test_auroc  test_auroc_full_img
backbone    modality                                               
dino_vits   colposcopy       0.537       0.468                0.467
            cytology         0.945       0.952                0.770
distilled   colposcopy       0.520       0.486                0.489
            cytology         0.892       0.908                0.651
resnet50    colposcopy       0.565       0.481                0.471
            cytology         0.910       0.929                0.784
vit_s_in21k colposcopy       0.548       0.480                0.467
            cytology         0.904       0.920                0.651
backbone     modality  
dino_vits    colposcopy      {'oc': 2, 'if': 1}
             cytology                {'knn': 3}
distilled    colposcopy               {'ae': 3}
             cytology                {'knn': 3}
resnet50     colposcop

In [ ]:
# 14. [NEW] Repeated group-level re-splitting (robustness to the single fixed split SEED=42).
# Frozen backbones only: the distilled student was trained on the ORIGINAL train split, so re-splitting it would leak.
def resplit(seed):
    sp = pd.Series("", index=meta.index, dtype=object)
    for _, g in meta[meta.label == 0].groupby("source"):
        for nm, d in zip(["train", "val", "lval", "test"], split_groups(g, [0.55, 0.15, 0.10, 0.20], seed)): sp[d.index] = nm
    for _, g in meta[meta.label == 1].groupby("source"):
        for nm, d in zip(["lval", "pool", "test"], split_groups(g, [0.20, 0.20, 0.60], seed)): sp[d.index] = nm
    return sp
orig_split = meta["split"].copy(); rs_rows = []
try:
    for s_ in (1, 2, 3, 4, 5):
        meta["split"] = resplit(s_).values
        for b in [k for k in FE if k != "distilled"]:
            for m in ["cytology", "colposcopy"]:
                Dm = mod_data(m); Z = {k: FE[b][v.index.values] for k, v in Dm.items()}
                bun = Bundle(seed=0).fit(Z["train"], Z["val"])
                mac = macro(evaluate(Dm["test"], bun.score(Z["test"]), ci_for=()))
                rs_rows.append(dict(split_seed=s_, backbone=b, modality=m, **mac.to_dict()))
finally:
    meta["split"] = orig_split
rs = pd.DataFrame(rs_rows); rs.to_csv(f"{SAVE}/resplit_robustness.csv", index=False)
print("Macro AUROC over 5 re-splits (mean / std):")
print(rs.groupby(["backbone", "modality"])[["img", "rec", "maha", "knn"]].agg(["mean", "std"]).round(3).to_string())


Macro AUROC over 5 re-splits (mean / std):
                          img           rec          maha           knn       
                         mean    std   mean    std   mean    std   mean    std
backbone    modality                                                          
dino_vits   colposcopy  0.461  0.016  0.521  0.010  0.519  0.014  0.529  0.018
            cytology    0.764  0.014  0.943  0.008  0.930  0.011  0.951  0.008
resnet50    colposcopy  0.486  0.009  0.499  0.008  0.494  0.012  0.497  0.014
            cytology    0.769  0.021  0.919  0.006  0.903  0.009  0.928  0.010
vit_s_in21k colposcopy  0.480  0.022  0.466  0.027  0.466  0.022  0.467  0.025
            cytology    0.661  0.016  0.913  0.008  0.905  0.008  0.921  0.012
